In [ ]:
!pip -q install -U sentence-transformers rank_bm25

In [ ]:
import json, glob, re, random, unicodedata, numpy as np, torch
from collections import defaultdict
from rank_bm25 import BM25Okapi
from sentence_transformers import SentenceTransformer, CrossEncoder

def find(name):
    hits = glob.glob(f"/kaggle/input/**/{name}", recursive=True)
    assert hits, f"{name} not found under /kaggle/input"
    return hits[0]

chunks = [json.loads(l) for l in open(find("chunks.jsonl"), encoding="utf-8")]
qa = [json.loads(l) for l in open(find("qa_multigold.jsonl"), encoding="utf-8")]
qa = [r for r in qa if r["kb_chunk_ids"]]
ids = [c["chunk_id"] for c in chunks]
src = {c["chunk_id"]: c["source_id"] for c in chunks}
N = len(ids)
fact = lambda r: r["notes"].split(";")[0]
facts = sorted({fact(r) for r in qa})
random.Random(42).shuffle(facts)
test_facts = set(facts[len(facts) // 2:])
print(f"KB v3: {N} chunks, {len(qa)} questions, {len(facts)} facts, {len(test_facts)} test facts")

# 1. Roman Urdu conversions reused from the e3_kb_v3 run
prev = json.load(open(find("e3_kb_v3.json"), encoding="utf-8"))
translit, bad = prev["transliterations"], prev["conversion_not_pure_urdu"]
print(f"conversions {len(translit)}, not pure Urdu {bad}")

# 2. Retrievers
FOLD = str.maketrans({"ي": "ی", "ى": "ی", "ك": "ک", "ە": "ہ", "ۀ": "ۂ"})
tok = lambda s: re.findall(r"\w+", unicodedata.normalize("NFKC", s).translate(FOLD).lower())
bm = BM25Okapi([tok(c["text"]) for c in chunks])
enc = SentenceTransformer("BAAI/bge-m3", device="cuda"); enc.max_seq_length = 1024
rer = CrossEncoder("BAAI/bge-reranker-v2-m3", device="cuda", max_length=1024)
C = enc.encode([c["text"] for c in chunks], batch_size=8, normalize_embeddings=True)
qtexts = sorted({r["question"] for r in qa} | set(translit.values()))
E = dict(zip(qtexts, enc.encode(qtexts, batch_size=32, normalize_embeddings=True)))
def rrf(*rows, k=60):
    f = np.zeros(N)
    for s in rows:
        for p, j in enumerate(np.argsort(-s)):
            f[j] += 1 / (k + p + 1)
    return f
def rerank(q, order):
    sc = rer.predict([(q, chunks[j]["text"]) for j in order[:20]], batch_size=8)
    return [order[i] for i in np.argsort(-sc)] + order[20:]
order = lambda s: [int(j) for j in np.argsort(-s)]
def query(r):
    return translit[r["id"]] if r["lang"] == "ur-Latn" else r["question"]
METHODS = {
    "BM25": lambda r: order(bm.get_scores(tok(r["question"]))),
    "BGE-M3": lambda r: order(C @ E[r["question"]]),
    "Ours (routed + conversion + reranker)": lambda r: rerank(query(r), order(C @ E[r["question"]]) if r["lang"] == "en"
                                                              else order(rrf(bm.get_scores(tok(query(r))), C @ E[query(r)]))),
}

# 3. Score on the unseen half, by language and by source
def score(rows, ranks, primary=False):
    r1, r5, mrr = [], [], []
    for r in rows:
        rk = ranks[r["id"]]
        gold = {r["kb_chunk_ids"][0]} if primary else set(r["kb_chunk_ids"])
        r1.append(ids[rk[0]] in gold); r5.append(any(ids[j] in gold for j in rk[:5]))
        mrr.append(next((1 / (p + 1) for p, j in enumerate(rk) if ids[j] in gold), 0.0))
    return {"n": len(rows), "r1": round(float(np.mean(r1)), 3), "r5": round(float(np.mean(r5)), 3), "mrr": round(float(np.mean(mrr)), 3)}
test = [r for r in qa if fact(r) in test_facts]
result = {}
for m, fn in METHODS.items():
    ranks = {r["id"]: fn(r) for r in test}
    for gold in ["one_gold", "all_gold"]:
        result.setdefault(gold, {})[m] = {lang: score([r for r in test if r["lang"] == lang], ranks, gold == "one_gold")
                                          for lang in ["ur", "ur-Latn", "en"]}
        print(f"{gold:9} {m:40} " + "  ".join(f"{l} R@5 {v['r5']}" for l, v in result[gold][m].items()))
by_src = defaultdict(list)
for r in test:
    by_src[src[r["kb_chunk_ids"][0]]].append(r)
ranks = {r["id"]: METHODS["Ours (routed + conversion + reranker)"](r) for r in test}
result["ours_by_source"] = {s: score(rows, ranks) for s, rows in sorted(by_src.items())}
print("\nOurs, R@5 by source (all languages):")
for s, v in result["ours_by_source"].items():
    print(f"  {s}  n={v['n']:3}  R@5 {v['r5']}")

json.dump({"kb": "v3", "gold": "qa_multigold.jsonl", "chunks": N, "questions": len(qa), "split": "facts shuffled seed 42, second half = test",
           "conversion_not_pure_urdu": bad, "results": result, "transliterations": translit},
          open("/kaggle/working/e3_kb_v3_multigold.json", "w", encoding="utf-8"), ensure_ascii=False, indent=1)
print("saved /kaggle/working/e3_kb_v3_multigold.json")
